In [ ]:
from langchain_postgres.vectorstores import PGVector

print("langchain_postgres import 성공")

In [ ]:
import os

from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_postgres.vectorstores import PGVector


load_dotenv(override=True)

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

connection_url = os.getenv("DATABASE_URL")

vector_store = PGVector(
    embeddings=embedding_model,
    collection_name="library_rules",
    connection=connection_url,
    use_jsonb=True,
)

print("PostgreSQL + pgvector 연결 및 초기화 완료")

In [ ]:
library_rule_chunks = [
    "[2026학년도 중앙도서관 대출 및 열람 규정]\n제1조(대출자격) 본교 재학생 및 휴학생은 학생증을 제시한 후 도서를 대출할 수 있습니다.",
    "제2조(대출 권수 및 기간) 학부생은 1인당 최대 5권까지 대출할 수 있으며 대출 기간은 14일입니다.\n단, 학술 연구용 특수 도서는 7일간 대출 가능하며 연장은 불가능합니다.",
    "제3조(연체 및 제재) 반납 기한을 초과할 경우 연체 일수만큼 대출이 정지됩니다."
]

chunk_ids=["rule_chunk_001", "rule_chunk_002", "rule_chunk_003"]

saved_ids = vector_store.add_texts(
    texts=library_rule_chunks,
    ids=chunk_ids
)

print(f"저장 갯수: {len(saved_ids)}")
print(f"청크id: {(saved_ids)}")


In [ ]:
# 사용자 질문을 정의합니다.
question = "특수 도서는 며칠 동안 대출할 수 있나요?"

search_results = vector_store.similarity_search(
    query=question,
    k=1  # 가장 유사한 1개의 결과를 반환
)   

print(search_results[0].page_content)

In [ ]:
search_results2 = vector_store.similarity_search_with_score(
    query=question, 
    k=3  # 가장 유사한 3개의 결과를 반환
)

for chunk, score in search_results2:
    print(f"청크 내용: {chunk.page_content}, 유사도 점수: {score:.4f}")   